In [3]:
import pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
import numpy as np
import os
from tqdm import tqdm
import gc
from numba import njit
import duckdb

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Define required columns based on usage in the notebook
required_columns = [
    'numHelped',
    'phase',
    'hasAnswer',
    'userId',
    'userFeNumHelped',
    'questionFeNumHelped',
    'numHelpProvidedAT',
    'hasHelped',
    'lnNumHelped',
    'numQuestionsAskedAT',
    'initialExperienceReceiving',
    'initialExperienceGiving'
]

df = pd.read_parquet('../data/study_datasets/question_centered_model_7d_processed.parquet',
                    columns=required_columns)
print(df.columns.tolist())

['numHelped', 'phase', 'hasAnswer', 'userId', 'userFeNumHelped', 'questionFeNumHelped', 'numHelpProvidedAT', 'hasHelped', 'lnNumHelped', 'numQuestionsAskedAT', 'initialExperienceReceiving', 'initialExperienceGiving']


# Investigate QE Value Error

In [7]:
# Ensure categorical types if they aren't already
df['phase'] = pd.Categorical(df['phase'])
df['hasAnswer'] = pd.Categorical(df['hasAnswer'])

print("--- Sample DataFrame Head ---")
print(df.head())
print("\n--- Value Counts for phase ---")
print(df['phase'].value_counts())
print("\n--- Value Counts for hasAnswer ---")
print(df['hasAnswer'].value_counts())

--- Sample DataFrame Head ---
   numHelped phase hasAnswer  userId  userFeNumHelped  questionFeNumHelped  \
0          1     1         1      48              0.0                 -3.0   
1          7     2         1      48              6.0                  3.0   
2          0     1         1      48             -1.0                  0.0   
3          0     2         1      48             -1.0                  0.0   
4          0     1         0      48             -1.0                  0.0   

   numHelpProvidedAT  hasHelped  lnNumHelped  numQuestionsAskedAT  \
0                  0          1     0.693147                    0   
1                  0          1     2.079442                    0   
2                 30          0     0.000000                    1   
3                 30          0     0.000000                    1   
4                 35          0     0.000000                    2   

  initialExperienceReceiving initialExperienceGiving  
0             no help seeked   

In [8]:
print("\n--- Cross-tabulation of phase and hasAnswer ---")
# Ensure this is the same 'df' used in your smf.ols() call
crosstab_results = pd.crosstab(df['phase'], df['hasAnswer'], margins=True, dropna=False)
print(crosstab_results)


--- Cross-tabulation of phase and hasAnswer ---
hasAnswer        0         1       All
phase                                 
1          3576461  20128089  23704550
2          3576461  20128089  23704550
All        7152922  40256178  47409100


# Basic Models with just phase interaction

In [4]:
import sys
import statsmodels.formula.api as smf
from statsmodels.stats.sandwich_covariance import cov_cluster
from IPython.display import HTML
import numpy as np
import pandas as pd
from stargazer.stargazer import Stargazer
sys.modules['stargazer.translators.statsmodels'].pd = pd

# Define the formula for the three models - fixed column names
formula1 = "numHelped ~ C(phase) + C(hasAnswer) + C(phase):C(hasAnswer)"
formula2 = "userFeNumHelped ~ C(phase) + C(hasAnswer) + C(phase):C(hasAnswer)"
formula3 = "questionFeNumHelped ~ C(phase) + C(hasAnswer) + C(phase):C(hasAnswer)"

# Model names
model_names = ["Base Model", "User FE", "Question FE"]
formulas = [formula1, formula2, formula3]

# Run and display each model individually
for i, (formula, name) in enumerate(zip(formulas, model_names)):
    print(f"\n\n==== {name} ====")

    # Fit the model
    model = smf.ols(formula=formula, data=df).fit()

    # Apply clustered standard errors by userId (fixed column name)
    model = model.get_robustcov_results(
        cov_type='cluster',
        groups=df['userId']
    )

    # Create a Stargazer table for this single model
    single_stargazer = Stargazer([model])
    single_stargazer.title(f"Effect of Receiving Answers on Providing Help - {name}")
    single_stargazer.significant_digits(3)
    single_stargazer.show_degrees_of_freedom(False)
    single_stargazer.show_model_numbers(False)

    # Display HTML output for this model
    html_output = single_stargazer.render_html()
    display(HTML(html_output))



==== Base Model ====




==== User FE ====




==== Question FE ====


C:\Users\svenp\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2
  warnings.warn('covariance of constraints does not have full '


# Models with numHelpProvided Buckets

In [5]:
# Distribution table showing users with at least X help up to 10
user_max_help = df.groupby('userId')['numHelpProvidedAT'].max().reset_index()
print("\nDistribution of Users with Max numHelpProvidedAT at least:")
total_users = len(user_max_help)
for threshold in range(11):  # 0 to 10
    users_with_at_least = (user_max_help['numHelpProvidedAT'] >= threshold).sum()
    percentage = users_with_at_least / total_users * 100
    print(f"  {threshold}: {users_with_at_least:,} users ({percentage:.1f}%)")

# Define cutoff and filter to users with max help >= cutoff
cutoff = 5
print(f"\nUsing a cutoff of numHelpProvidedAT >= {cutoff}")
included_users = user_max_help[user_max_help['numHelpProvidedAT'] >= cutoff]['userId'].tolist()
filtered_by_user_df = df[df['userId'].isin(included_users)].copy()
print(f"\nFiltered to {len(included_users):,} users who have provided at least {cutoff} help at least once")
print(f"Events after user filtering: {len(filtered_by_user_df):,} ({len(filtered_by_user_df)/len(df):.1%} of original data)")
filtered_df = filtered_by_user_df[filtered_by_user_df['numHelpProvidedAT'] <= cutoff].copy()
print(f"Events after max help filtering: {len(filtered_df):,} ({len(filtered_df)/len(filtered_by_user_df):.1%} of user-filtered data)")
filtered_df['numHelpProvidedAT_binned'] = filtered_df['numHelpProvidedAT']

# Re-calculate user FEs
filtered_df['userFeNumHelped'] = filtered_df.groupby("userId")["numHelped"].transform(lambda x: x - x.mean())
filtered_df['userFeHasHelped'] = filtered_df.groupby("userId")["hasHelped"].transform(lambda x: x - x.mean())
filtered_df['userFeLnNumHelped'] = filtered_df.groupby("userId")["lnNumHelped"].transform(lambda x: x - x.mean())

# Show distribution of events by bin
bin_counts = filtered_df['numHelpProvidedAT_binned'].value_counts().sort_index()
print("\nDistribution of numHelpProvidedAT_binned in filtered dataset:")
for bin_val, count in bin_counts.items():
    percentage = count / len(filtered_df) * 100
    print(f"  {bin_val}: {count:,} events ({percentage:.1f}%)")

# Define formulas with bin interactions (0 is reference level)
formula1 = """numHelped ~ C(phase) + C(hasAnswer) + C(numHelpProvidedAT_binned) + C(phase):C(hasAnswer) +
              C(phase):C(hasAnswer):C(numHelpProvidedAT_binned)"""

formula2 = """userFeNumHelped ~ C(phase) + C(hasAnswer) + C(numHelpProvidedAT_binned) + C(phase):C(hasAnswer) +
              C(phase):C(hasAnswer):C(numHelpProvidedAT_binned)"""

formula3 = """questionFeNumHelped ~ C(phase) + C(hasAnswer) + C(numHelpProvidedAT_binned) + C(phase):C(hasAnswer) +
              C(phase):C(hasAnswer):C(numHelpProvidedAT_binned)"""

model_names = ["Base Model", "User FE", "Question FE"]
formulas = [formula1, formula2, formula3]

# Run and display each model with clustered standard errors
for i, (formula, name) in enumerate(zip(formulas, model_names)):
    print(f"\n\n==== {name} ====")
    print(f"Formula: {formula}")

    model = smf.ols(formula=formula, data=filtered_df).fit()
    model = model.get_robustcov_results(
        cov_type='cluster',
        groups=filtered_df['userId']
    )

    single_stargazer = Stargazer([model])
    single_stargazer.title(f"Effect of Receiving Answers on Providing Help - {name}")
    single_stargazer.significant_digits(3)
    single_stargazer.show_degrees_of_freedom(False)
    single_stargazer.show_model_numbers(False)

    html_output = single_stargazer.render_html()
    display(HTML(html_output))


Distribution of Users with Max numHelpProvidedAT at least:
  0: 4,995,121 users (100.0%)
  1: 945,002 users (18.9%)
  2: 661,254 users (13.2%)
  3: 536,132 users (10.7%)
  4: 459,930 users (9.2%)
  5: 406,009 users (8.1%)
  6: 364,797 users (7.3%)
  7: 332,685 users (6.7%)
  8: 305,737 users (6.1%)
  9: 283,194 users (5.7%)
  10: 263,784 users (5.3%)

Using a cutoff of numHelpProvidedAT >= 5

Filtered to 406,009 users who have provided at least 5 help at least once
Events after user filtering: 15,825,110 (33.4% of original data)
Events after max help filtering: 5,815,644 (36.7% of user-filtered data)

Distribution of numHelpProvidedAT_binned in filtered dataset:
  0: 2,191,044 events (37.7%)
  1: 872,844 events (15.0%)
  2: 693,280 events (11.9%)
  3: 630,512 events (10.8%)
  4: 633,394 events (10.9%)
  5: 794,570 events (13.7%)


==== Base Model ====
Formula: numHelped ~ C(phase) + C(hasAnswer) + C(numHelpProvidedAT_binned) + C(phase):C(hasAnswer) +
              C(phase):C(hasAnswer



==== User FE ====
Formula: userFeNumHelped ~ C(phase) + C(hasAnswer) + C(numHelpProvidedAT_binned) + C(phase):C(hasAnswer) +
              C(phase):C(hasAnswer):C(numHelpProvidedAT_binned)




==== Question FE ====
Formula: questionFeNumHelped ~ C(phase) + C(hasAnswer) + C(numHelpProvidedAT_binned) + C(phase):C(hasAnswer) +
              C(phase):C(hasAnswer):C(numHelpProvidedAT_binned)


C:\Users\svenp\AppData\Local\Programs\Python\Python313\Lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 23, but rank is 12
  warnings.warn('covariance of constraints does not have full '


# Models with experience receiving/providing

In [6]:
import sys
import statsmodels.formula.api as smf
from IPython.display import HTML
import pandas as pd
import numpy as np
from stargazer.stargazer import Stargazer
sys.modules['stargazer.translators.statsmodels'].pd = pd

# Model 1: Users with both numQuestionsAskedAT=0 and numQuestionsAskedAT=1
# First identify users who have both values
users_with_0 = set(df[df['numQuestionsAskedAT'] == 0]['userId'])
users_with_1 = set(df[df['numQuestionsAskedAT'] == 1]['userId'])
users_with_both = users_with_0.intersection(users_with_1)

print(f"Users with both values: {len(users_with_both)}")

# Filter dataset to only include these users and where numQuestionsAskedAT is 0 or 1
filtered_df1 = df[
    (df['userId'].isin(users_with_both)) &
    (df['numQuestionsAskedAT'].isin([0, 1]))
].copy()

print(f"Distribution of initialExperienceReceiving values:")
print(filtered_df1['initialExperienceReceiving'].value_counts())

# Convert string variables to categorical type
filtered_df1['initialExperienceReceiving'] = filtered_df1['initialExperienceReceiving'].astype('category')
filtered_df1['hasAnswer'] = filtered_df1['hasAnswer'].astype(int)  # Make sure this is numeric

# Formula for Model 1
formula1 = 'questionFeNumHelped ~ phase * C(initialExperienceReceiving) + phase:C(initialExperienceReceiving):C(hasAnswer)'

# Fit Model 1
model1 = smf.ols(formula=formula1, data=filtered_df1).fit()

# Apply clustered standard errors by userId
model1 = model1.get_robustcov_results(
    cov_type='cluster',
    groups=filtered_df1['userId']
)

# Create Stargazer table for this model
model1_stargazer = Stargazer([model1])
model1_stargazer.title("Model 1: Effect of initialExperienceReceiving on Helping")
model1_stargazer.significant_digits(3)
model1_stargazer.show_degrees_of_freedom(False)
model1_stargazer.show_model_numbers(False)

# Display HTML output for Model 1
html_output1 = model1_stargazer.render_html()
display(HTML(html_output1))

# Model 2: Users with numQuestionsAskedAT > 0 and both numHelpProvidedAT=0 and numHelpProvidedAT=1
# First identify users who have both values of numHelpProvidedAT among those with numQuestionsAskedAT > 0
questioned_users = df[df['numQuestionsAskedAT'] > 0]['userId'].unique()
help0_users = set(df[(df['numQuestionsAskedAT'] > 0) & (df['numHelpProvidedAT'] == 0)]['userId'])
help1_users = set(df[(df['numQuestionsAskedAT'] > 0) & (df['numHelpProvidedAT'] == 1)]['userId'])
users_with_both_help = help0_users.intersection(help1_users)

print(f"Users with both values: {len(users_with_both_help)}")

# Filter dataset for Model 2
filtered_df2 = df[
    (df['userId'].isin(users_with_both_help)) &
    (df['numQuestionsAskedAT'] > 0) &
    (df['numHelpProvidedAT'].isin([0, 1]))
].copy()

print(f"Distribution of initialExperienceGiving values:")
print(filtered_df2['initialExperienceGiving'].value_counts())

# Convert string variables to categorical type for model 2
filtered_df2['initialExperienceGiving'] = filtered_df2['initialExperienceGiving'].astype('category')
filtered_df2['hasAnswer'] = filtered_df2['hasAnswer'].astype(int)

# Formula for Model 2
formula2 = 'questionFeNumHelped ~ phase * C(initialExperienceGiving) + phase:C(initialExperienceGiving):C(hasAnswer)'

# Fit Model 2
model2 = smf.ols(formula=formula2, data=filtered_df2).fit()

# Apply clustered standard errors by userId
model2 = model2.get_robustcov_results(
    cov_type='cluster',
    groups=filtered_df2['userId']
)

# Create Stargazer table for Model 2
model2_stargazer = Stargazer([model2])
model2_stargazer.title("Model 2: Effect of initialExperienceGiving on Helping")
model2_stargazer.significant_digits(3)
model2_stargazer.show_degrees_of_freedom(False)
model2_stargazer.show_model_numbers(False)

# Display HTML output for Model 2
html_output2 = model2_stargazer.render_html()
display(HTML(html_output2))

Users with both values: 1900458
Distribution of initialExperienceReceiving values:
initialExperienceReceiving
no help seeked    4069996
help seeked       2392306
help received     2138138
Name: count, dtype: Int64


Users with both values: 202989
Distribution of initialExperienceGiving values:
initialExperienceGiving
no help attempted    3044010
help attempted       2089874
helped                356920
Name: count, dtype: Int64
